<a href="https://colab.research.google.com/github/tarlie18/Agentic_AI_103/blob/main/agentic_ai_103.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U langgraph langchain langchain-google-genai google-genai

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from getpass import getpass

google_api_key = getpass("Enter your Gemini API key: ")
if not google_api_key:
    raise ValueError("No API key entered — rerun this cell and paste your key.")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key,
    max_retries=0
)

In [ ]:
from typing import TypedDict, Optional

class AgentState(TypedDict):
    topic: str
    research_notes: str
    current_draft: str
    reviewer_feedback: str
    approved: bool
    human_decision: Optional[str]
    revision_count: int

In [ ]:
RESEARCHER_SYSTEM = """You are an HHS Epidemiologist. Your job is to compile a short,
factual bulleted list of the key data points about a disease outbreak: pathogen, likely source,
symptoms, at-risk groups, and recommended precautions. Be concise and factual. Do not draft any
public-facing memo — that is not your job. Output ONLY the bulleted facts."""

def research_node(state: AgentState) -> dict:
    response = llm.invoke([
        ("system", RESEARCHER_SYSTEM),
        ("human", f"Topic: {state['topic']}"),
    ])
    return {"research_notes": response.content}

In [ ]:
WRITER_SYSTEM = """You are a Public Health Communications Officer at HHS. Draft an
official public health advisory memo based ONLY on the facts you are given. Use a clear,
authoritative, calm tone appropriate for an FDA/HHS advisory. Include a headline, a summary,
and recommended public actions. If reviewer feedback is provided, revise the draft to address it
directly."""

def writer_node(state: AgentState) -> dict:
    feedback = state.get("reviewer_feedback", "")
    prompt = f"""Research notes:
{state['research_notes']}

Reviewer feedback (address this if not empty): {feedback or 'None yet — this is the first draft.'}
"""
    response = llm.invoke([
        ("system", WRITER_SYSTEM),
        ("human", prompt),
    ])
    return {
        "current_draft": response.content,
        "revision_count": state.get("revision_count", 0) + 1,
    }

In [ ]:
REVIEWER_SYSTEM = """You are a Compliance Officer for HHS. Compare the memo draft against
the research notes. Check for any claim in the draft NOT supported by the research notes
(a hallucination), tone problems, or missing recommended actions.

Respond in EXACTLY this format:
VERDICT: APPROVE or REVISE
FEEDBACK: <one or two sentences of specific, actionable feedback. If APPROVE, just say "Looks accurate and complete.">"""

def reviewer_node(state: AgentState) -> dict:
    prompt = f"""Research notes:
{state['research_notes']}

Draft memo:
{state['current_draft']}
"""
    response = llm.invoke([
        ("system", REVIEWER_SYSTEM),
        ("human", prompt),
    ])
    text = response.content
    verdict_line = next((l for l in text.splitlines() if l.upper().startswith("VERDICT")), "VERDICT: REVISE")
    feedback_line = next((l for l in text.splitlines() if l.upper().startswith("FEEDBACK")), "FEEDBACK: (none parsed)")
    approved = "APPROVE" in verdict_line.upper()
    return {
        "approved": approved,
        "reviewer_feedback": feedback_line,
    }

In [ ]:
from langgraph.types import interrupt

def human_approval_node(state: AgentState) -> dict:
    decision = interrupt({
        "instruction": "Review this advisory memo. Reply with one of: "
                        "'approve', 'edit: <your replacement text>', or 'steer: <your instruction>'.",
        "draft": state["current_draft"],
        "reviewer_feedback": state.get("reviewer_feedback", ""),
    })
    return {"human_decision": decision}

In [ ]:
def route_after_review(state: AgentState) -> str:
    if state.get("approved"):
        return "Human_Approval"
    if state.get("revision_count", 0) >= 3:
        return "Human_Approval"
    return "Writer"

def route_after_human(state: AgentState) -> str:
    decision = (state.get("human_decision") or "").strip().lower()
    if decision.startswith("approve"):
        return "END"
    if decision.startswith("edit:"):
        return "END"
    if decision.startswith("steer:"):
        return "Writer"
    return "Writer"

In [ ]:
def apply_human_decision(state: AgentState) -> dict:
    decision = state.get("human_decision", "") or ""
    lowered = decision.strip().lower()
    if lowered.startswith("edit:"):
        return {"current_draft": decision.split(":", 1)[1].strip(), "approved": True}
    if lowered.startswith("steer:"):
        return {"reviewer_feedback": decision.split(":", 1)[1].strip(), "approved": False}
    if lowered.startswith("approve"):
        return {"approved": True}
    return {}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(AgentState)

workflow.add_node("Researcher", research_node)
workflow.add_node("Writer", writer_node)
workflow.add_node("Reviewer", reviewer_node)
workflow.add_node("Human_Approval", human_approval_node)
workflow.add_node("Apply_Decision", apply_human_decision)

workflow.add_edge(START, "Researcher")
workflow.add_edge("Researcher", "Writer")
workflow.add_edge("Writer", "Reviewer")

workflow.add_conditional_edges(
    "Reviewer",
    route_after_review,
    {"Writer": "Writer", "Human_Approval": "Human_Approval"},
)

workflow.add_edge("Human_Approval", "Apply_Decision")

workflow.add_conditional_edges(
    "Apply_Decision",
    route_after_human,
    {"Writer": "Writer", "END": END},
)

checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

print("Graph compiled. Nodes:", list(graph.get_graph().nodes))

In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "advisory-demo-1"}}

result = graph.invoke(
    {
        "topic": "a localized E. coli outbreak",
        "research_notes": "",
        "current_draft": "",
        "reviewer_feedback": "",
        "approved": False,
        "human_decision": None,
        "revision_count": 0,
    },
    config=config,
)

print(result.get("__interrupt__"))

In [ ]:
human_response = "steer: make the tone more urgent and add a specific call-to-action"

result = graph.invoke(Command(resume=human_response), config=config)

if result.get("__interrupt__"):
    print("--- Graph paused again for your review ---")
    print(result["__interrupt__"])
else:
    print("--- Workflow finished ---")
    print(result["current_draft"])

In [ ]:
final_result = graph.invoke(Command(resume="approve"), config=config)

if not final_result.get("__interrupt__"):
    print("=== FINAL APPROVED ADVISORY ===\n")
    print(final_result["current_draft"])
else:
    print("Still paused — provide another decision.")
    print(final_result["__interrupt__"])

In [ ]:
from langgraph.types import Command
import uuid

def chat_with_advisory_team(topic: str, thread_id: str = None):
    """Runs the Researcher->Writer->Reviewer team on `topic`, then keeps the
    conversation open at the human checkpoint until you type 'approve'."""
    thread_id = thread_id or str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    result = graph.invoke(
        {
            "topic": topic,
            "research_notes": "",
            "current_draft": "",
            "reviewer_feedback": "",
            "approved": False,
            "human_decision": None,
            "revision_count": 0,
        },
        config=config,
    )

    while result.get("__interrupt__"):
        payload = result["__interrupt__"][0].value
        print("\n--- DRAFT FOR YOUR REVIEW ---")
        print(payload["draft"])
        print("\n(reviewer notes:", payload.get("reviewer_feedback", ""), ")")

        user_input = input(
            "\nYour response ('approve', 'edit: ...', 'steer: ...', or 'quit'): "
        )
        if user_input.strip().lower() == "quit":
            print("Session left open — resume later with the same thread_id:", thread_id)
            return thread_id
        result = graph.invoke(Command(resume=user_input), config=config)

    print("\n=== FINAL APPROVED ADVISORY ===\n")
    print(result["current_draft"])
    return thread_id

In [ ]:
chat_with_advisory_team("a foodborne illness at a school cafeteria")